# Spark DataFrame 가공

- 컬럼 추가/변경/삭제, 결측치(null, NaN) 처리, 사용자 정의 함수(UDF)
- Spark DataFrame은 수정 불가(immutable): `withColumn`, `drop`, `fillna` 등은 **항상 새 DataFrame을 반환**
- 그래서 결과를 쓰려면 변수에 다시 담아야 함: pandas `df["A"] = ...` / Spark `df = df.withColumn("A", ...)`

## 0. Setup

- `spark.sql.execution.arrow.pyspark.enabled`: pandas <-> Spark 변환(`createDataFrame`, `toPandas`)에 Arrow 사용
- Spark 3.5(이 Docker 이미지)는 기본값 false, 최신 Spark는 기본값 true. 켜야 pandas의 NaN이 Spark에서 null로 바뀜 (5절 참고)
- `.config()`는 builder 메서드라서 **`.getOrCreate()` 앞에** 써야 함 (뒤에 쓰면 AttributeError)

In [32]:
import pandas as pd, numpy as np
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F  # F.count, F.max, F.upper ...
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
        .appName("SparkML")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")  # pandas NaN -> Spark null
        .enableHiveSupport()
        .getOrCreate()
)

DATA_PATH = Path("./data")

26/10/16 01:36:47 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 1. 데이터 로드

In [2]:
titanic_sdf = spark.read.csv(str(DATA_PATH / "titanic_train.csv"), header=True, inferSchema=True)
titanic_sdf = titanic_sdf.cache()     # T
titanic_pdf = titanic_sdf.toPandas()  # A: pandas 비교용

In [3]:
titanic_pdf.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,None,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,None,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,None,S


In [4]:
titanic_sdf.limit(5).show()  # CSV의 빈 칸은 Spark에서 NULL, pandas에서 None

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

In [5]:
# 원본을 보존하고 실습용 복사본 만들기
titanic_pdf_2 = titanic_pdf.copy()        # pandas: 복사하지 않으면 원본이 바뀜
titanic_sdf_2 = titanic_sdf.select("*")   # Spark: 원래 immutable이라 그대로 써도 원본은 안 바뀜

## 2. 컬럼 추가/변경 (withColumn) - T

| 목적 | pandas | Spark |
|---|---|---|
| 새 컬럼 추가 | `df["B"] = df["A"] * 10` | `withColumn("B", col("A") * 10)` |
| 기존 컬럼 값 변경 | `df["A"] = df["A"] + 20` | `withColumn("A", col("A") + 20)` (같은 이름이면 덮어씀) |
| 타입 변경 | `df["A"].astype(np.int64)` | `withColumn("A", col("A").cast("Integer"))` |
| 상수 컬럼 | `df["B"] = 10` | `withColumn("B", lit(10))` |

- `withColumn`의 두 번째 인자는 **반드시 Column**. 상수는 `lit()`으로 감싸야 함 (그냥 `10`을 넣으면 NOT_COLUMN 오류)
- `withColumn`은 체이닝 가능하고 왼쪽부터 순서대로 적용됨
- `cast("Integer")`는 소수점을 버림 (27.25 -> 27)

In [6]:
titanic_pdf_2["Extra_Fare"] = titanic_pdf_2["Fare"] * 10
titanic_sdf_2 = titanic_sdf_2.withColumn("Extra_Fare", col("Fare") * 10)

titanic_sdf_2.limit(5).show()
titanic_pdf_2.head()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+----------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|Extra_Fare|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+----------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|      72.5|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|   712.833|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|     79.25|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|     531.0|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|    

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Extra_Fare
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,None,S,72.500
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,712.833
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,None,S,79.250
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,531.000
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,None,S,80.500


In [7]:
titanic_pdf_2["Fare"] = titanic_pdf_2["Fare"] + 20
titanic_sdf_2 = titanic_sdf_2.withColumn("Fare", col("Fare") + 20)

titanic_sdf_2.limit(5).show()
titanic_pdf_2.head()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+----------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|Extra_Fare|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+----------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|  27.25| NULL|       S|      72.5|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|91.2833|  C85|       C|   712.833|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282| 27.925| NULL|       S|     79.25|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   73.1| C123|       S|     531.0|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|  28.05| NULL|       S|    

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Extra_Fare
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,27.2500,None,S,72.500
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,91.2833,C85,C,712.833
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,27.9250,None,S,79.250
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,73.1000,C123,S,531.000
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,28.0500,None,S,80.500


In [8]:
titanic_pdf_2["Fare"] = titanic_pdf_2["Fare"].astype(np.int64)
titanic_sdf_2 = titanic_sdf_2.withColumn("Fare", col("Fare").cast("Integer"))

titanic_sdf_2.limit(5).show()
titanic_pdf_2.head()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+----+-----+--------+----------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|Fare|Cabin|Embarked|Extra_Fare|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+----+-----+--------+----------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|  27| NULL|       S|      72.5|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|  91|  C85|       C|   712.833|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  27| NULL|       S|     79.25|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|  73| C123|       S|     531.0|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|  28| NULL|       S|      80.5|
+-----------+---

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Extra_Fare
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,27,None,S,72.500
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,91,C85,C,712.833
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,27,None,S,79.250
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,73,C123,S,531.000
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,28,None,S,80.500


In [9]:
# 체이닝: 왼쪽부터 순서대로 적용
# titanic_sdf_2는 위에서 이미 +20, cast가 적용된 상태라 Fare가 한 번 더 늘어남
(
    titanic_sdf_2
    .withColumn("Extra_Fare", col("Fare") * 10)
    .withColumn("Fare", col("Fare") + 20)
    .withColumn("Fare", col("Fare").cast("Integer"))
    .limit(5)
    .show()
)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+----+-----+--------+----------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|Fare|Cabin|Embarked|Extra_Fare|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+----+-----+--------+----------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|  47| NULL|       S|       270|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599| 111|  C85|       C|       910|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  47| NULL|       S|       270|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|  93| C123|       S|       730|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|  48| NULL|       S|       280|
+-----------+---

In [10]:
titanic_sdf_2 = titanic_sdf_2.withColumn("Extra_Fare", 10)  # 오류: 상수는 Column이 아님 -> lit() 필요

PySparkTypeError: [NOT_COLUMN] Argument `col` should be a Column, got int.

In [12]:
from pyspark.sql.functions import lit

titanic_sdf_2 = titanic_sdf_2.withColumn("Extra_Fare", lit(10))
titanic_sdf_2 = titanic_sdf_2.withColumn("New_Name", lit("Test_name"))

titanic_sdf_2.limit(5).show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+----+-----+--------+----------+---------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|Fare|Cabin|Embarked|Extra_Fare| New_Name|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+----+-----+--------+----------+---------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|  27| NULL|       S|        10|Test_name|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|  91|  C85|       C|        10|Test_name|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  27| NULL|       S|        10|Test_name|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|  73| C123|       S|        10|Test_name|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0| 

## 3. select로 컬럼 만들기 / 문자열 함수 / 이름 변경 - T

| 목적 | pandas | Spark |
|---|---|---|
| 컬럼 복사해서 새 이름 | `df["Gender"] = df["Sex"]` | `select("*", col("Sex").alias("Gender"))` 또는 `withColumn("Gender", col("Sex"))` |
| 문자열 자르기 | `df["A"].str[0:1]` | `substring(col("A"), 1, 1)` |
| 구분자로 나누기 | `df["A"].str.split(",").str[0]` | `split(col("A"), ",").getItem(0)` |
| 컬럼 이름 변경 | `rename(columns={"old": "new"})` | `withColumnRenamed("old", "new")` |

- `select("*", 식.alias(...))`와 `withColumn(...)`은 결과가 같음
- `substring(col, pos, len)`의 pos는 **1부터 시작** (0을 넣어도 1처럼 동작)
- null에 문자열 함수를 적용하면 결과도 null (Cabin이 null이면 Cabin_First도 null)
- `split(...).getItem(1)`은 앞 공백이 남음 (`" Mr. Owen Harris"`). 필요하면 `F.trim()`
- 컬럼이 많을 땐 `display(sdf.limit(5).toPandas())`가 보기 편함 (작은 결과에만 사용)

In [13]:
from pyspark.sql.functions import substring

# select + alias
titanic_sdf_2 = titanic_sdf_2.select("*", col("Sex").alias("Gender"))
titanic_sdf_2 = titanic_sdf_2.select("*", substring(col("Cabin"), 0, 1).alias("Cabin_First"))

# withColumn (결과 동일)
titanic_sdf_2 = titanic_sdf_2.withColumn("Gender2", col("Sex")).withColumn("Cabin_First2", substring(col("Cabin"), 0, 1))

display(titanic_sdf_2.limit(5).toPandas())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Extra_Fare,New_Name,Gender,Cabin_First,Gender2,Cabin_First2
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,27,None,S,10,Test_name,male,None,male,None
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,91,C85,C,10,Test_name,female,C,female,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,27,None,S,10,Test_name,female,None,female,None
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,73,C123,S,10,Test_name,female,C,female,C
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,28,None,S,10,Test_name,male,None,male,None


In [14]:
from pyspark.sql.functions import split

titanic_sdf_2 = titanic_sdf_2.withColumn("Name1", split(col("Name"), ",").getItem(0))  # 성
titanic_sdf_2 = titanic_sdf_2.withColumn("Name2", split(col("Name"), ",").getItem(1))  # 호칭 + 이름 (앞 공백 포함)

display(titanic_sdf_2.limit(5).toPandas())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Extra_Fare,New_Name,Gender,Cabin_First,Gender2,Cabin_First2,Name1,Name2
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,27,None,S,10,Test_name,male,None,male,None,Braund,Mr. Owen Harris
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,91,C85,C,10,Test_name,female,C,female,C,Cumings,Mrs. John Bradley (Florence Briggs Thayer)
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,27,None,S,10,Test_name,female,None,female,None,Heikkinen,Miss. Laina
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,73,C123,S,10,Test_name,female,C,female,C,Futrelle,Mrs. Jacques Heath (Lily May Peel)
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,28,None,S,10,Test_name,male,None,male,None,Allen,Mr. William Henry


In [15]:
titanic_sdf_2 = titanic_sdf_2.withColumnRenamed("Gender", "Gender_Renamed")

display(titanic_sdf_2.limit(5).toPandas())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Extra_Fare,New_Name,Gender_Renamed,Cabin_First,Gender2,Cabin_First2,Name1,Name2
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,27,None,S,10,Test_name,male,None,male,None,Braund,Mr. Owen Harris
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,91,C85,C,10,Test_name,female,C,female,C,Cumings,Mrs. John Bradley (Florence Briggs Thayer)
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,27,None,S,10,Test_name,female,None,female,None,Heikkinen,Miss. Laina
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,73,C123,S,10,Test_name,female,C,female,C,Futrelle,Mrs. Jacques Heath (Lily May Peel)
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,28,None,S,10,Test_name,male,None,male,None,Allen,Mr. William Henry


## 4. 컬럼 삭제 (drop) - T

| pandas | Spark |
|---|---|
| `drop("A", axis=1)` 또는 `drop(columns=["A"])` | `drop("A")` 또는 `drop(col("A"))` |
| `drop(columns=["A", "B"])` | `drop("A", "B")` 또는 `drop(col("A"), col("B"))` |
| `drop(columns=cols)` | `drop(*cols)` (list는 unpack) |

- Spark에는 `axis`, `inplace`가 없음. 항상 새 DataFrame 반환
- 없는 컬럼명을 문자열로 넣으면 오류 없이 무시됨

In [16]:
# pandas 비교
titanic_pdf_d = titanic_pdf.drop("Name", axis=1, inplace=False)
titanic_pdf_d.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,male,22.0,1,0,A/5 21171,7.2500,None,S
1,2,1,1,female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,female,26.0,0,0,STON/O2. 3101282,7.9250,None,S
3,4,1,1,female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,male,35.0,0,0,373450,8.0500,None,S


In [17]:
titanic_sdf_3 = titanic_sdf.select("*")
titanic_sdf_d = titanic_sdf_3.drop(col("Sex"))

titanic_sdf_d.limit(5).show()

+-----------+--------+------+--------------------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+----+-----+-----+----------------+-------+-----+--------+



In [18]:
# 여러 컬럼 삭제: 문자열 / Column 둘 다 가능
titanic_sdf_3.drop("Age", "SibSp").limit(5).show()
titanic_sdf_3.drop(col("Age"), col("SibSp")).limit(5).show()

+-----------+--------+------+--------------------+------+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+-----+----------------+-------+-----+--------+

+-----------+--------+------+--------------------+------+-----+

In [19]:
# 조건으로 컬럼 고르기 + unpack (문자열 컬럼 전부 삭제)
drop_str_cols = [
    col_name
    for col_name, col_type in titanic_sdf_3.dtypes
    if col_type == "string"
]

titanic_sdf_3.drop(*drop_str_cols).limit(5).show()

+-----------+--------+------+----+-----+-----+-------+
|PassengerId|Survived|Pclass| Age|SibSp|Parch|   Fare|
+-----------+--------+------+----+-----+-----+-------+
|          1|       0|     3|22.0|    1|    0|   7.25|
|          2|       1|     1|38.0|    1|    0|71.2833|
|          3|       1|     3|26.0|    0|    0|  7.925|
|          4|       1|     1|35.0|    1|    0|   53.1|
|          5|       0|     3|35.0|    0|    0|   8.05|
+-----------+--------+------+----+-----+-----+-------+



## 5. 결측치 개념: null vs NaN

| 구분 | null | NaN |
|---|---|---|
| 의미 | 값이 없음 | "숫자가 아님"이라는 실수 값 (double/float 컬럼에만 존재) |
| 확인 | `col("A").isNull()`, `isNotNull()` | `F.isnan(col("A"))` |
| 예 | CSV의 빈 칸, Python `None` | `float("nan")`, 0.0/0.0 |

- pandas는 둘을 구분하지 않고 NaN, None 모두 결측치로 봄. `float("nan")`과 `np.nan`도 pandas 안에서는 같은 값
- pandas -> Spark 변환(`createDataFrame(pdf)`) 시
  - Arrow 켜짐: NaN -> **null**
  - Arrow 꺼짐(Spark 3.5 기본값): NaN -> **NaN** 그대로
- 그래서 Arrow를 켜면 pandas를 거쳐서는 NaN을 만들 수 없음 -> NaN 실습은 **리스트 + 스키마로 직접 생성**
- `isNull()`은 NaN을 못 잡음. 둘 다 찾으려면 `F.isnan(c) | col(c).isNull()` (01 노트북 참고)

In [33]:
# Arrow 켜짐: pandas의 NaN이 Spark에서 null로 바뀜
null_ex_pdf = pd.DataFrame(
    {
        "x": [1, np.nan],
        "y": [None, "foo"]
    }
)
null_ex_sdf = spark.createDataFrame(null_ex_pdf)
null_ex_sdf.show()

+----+----+
|   x|   y|
+----+----+
| 1.0|NULL|
|NULL| foo|
+----+----+



In [50]:
# pandas를 거치지 않으면 NaN과 null을 한 컬럼에 같이 둘 수 있음
null_ex_sdf2 = spark.createDataFrame(
    [(1.0, None), (float("nan"), "foo"), (None, "bar")],
    "x double, y string"
)
null_ex_sdf2.show()

+----+----+
|   x|   y|
+----+----+
| 1.0|NULL|
| NaN| foo|
|NULL| bar|
+----+----+



In [ ]:
# isnan은 NaN만, isNull은 null만 True
null_ex_sdf2.select(
    "x",
    F.isnan("x").alias("isnan"),
    col("x").isNull().alias("isnull"),
).show()

In [ ]:
# 참고: Arrow를 잠깐 끄면 pandas의 NaN이 NaN 그대로 들어감
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
spark.createDataFrame(null_ex_pdf).show()
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

## 6. 결측치 찾기 / 삭제

| 목적 | pandas | Spark |
|---|---|---|
| 결측 행 보기 | `df[df["Age"].isna()]` | `filter(col("Age").isNull())` |
| 컬럼별 결측 여부 | `isna()` | `select([col(c).isNull().alias(c) for c in cols])` |
| 컬럼별 결측 개수 | `isna().sum()` | `select([count(when(col(c).isNull(), c)).alias(c) for c in cols])` |
| 결측 행 전체 삭제 | `dropna()` | `dropna()` 또는 `na.drop()` |
| 특정 컬럼 기준 삭제 | `dropna(subset=[...])` | `na.drop(subset=[...])` |

- `count()`는 null을 세지 않음. `when(조건, 값)`은 조건이 False면 null -> 둘을 합치면 "조건을 만족하는 개수"
- `dropna()` / `na.drop()`은 null과 NaN을 **둘 다** 삭제
- `na.drop(subset=[...])`은 `filter(col(...).isNotNull() & ...)`와 같은 결과 (단, filter는 NaN을 못 거름)

In [34]:
titanic_sdf_3.filter(col("Age").isNull()).limit(10).show(10)

+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|  Ticket|    Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+-----+--------+
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|  330877|  8.4583| NULL|       Q|
|         18|       1|     2|Williams, Mr. Cha...|  male|NULL|    0|    0|  244373|    13.0| NULL|       S|
|         20|       1|     3|Masselmani, Mrs. ...|female|NULL|    0|    0|    2649|   7.225| NULL|       C|
|         27|       0|     3|Emir, Mr. Farred ...|  male|NULL|    0|    0|    2631|   7.225| NULL|       C|
|         29|       1|     3|"O'Dwyer, Miss. E...|female|NULL|    0|    0|  330959|  7.8792| NULL|       Q|
|         30|       0|     3| Todoroff, Mr. Lalio|  male|NULL|    0|    0|  349216|  7.8958| NULL|       S|
|         32|       1|     1

In [38]:
# 컬럼별 null 여부 (True/False)
titanic_sdf.select([col(col_name).isNull().alias(col_name) for col_name in titanic_sdf.columns]).limit(10).show()

[Stage 64:>                                                         (0 + 1) / 1]

+-----------+--------+------+-----+-----+-----+-----+-----+------+-----+-----+--------+
|PassengerId|Survived|Pclass| Name|  Sex|  Age|SibSp|Parch|Ticket| Fare|Cabin|Embarked|
+-----------+--------+------+-----+-----+-----+-----+-----+------+-----+-----+--------+
|      false|   false| false|false|false|false|false|false| false|false| true|   false|
|      false|   false| false|false|false|false|false|false| false|false|false|   false|
|      false|   false| false|false|false|false|false|false| false|false| true|   false|
|      false|   false| false|false|false|false|false|false| false|false|false|   false|
|      false|   false| false|false|false|false|false|false| false|false| true|   false|
|      false|   false| false|false|false| true|false|false| false|false| true|   false|
|      false|   false| false|false|false|false|false|false| false|false|false|   false|
|      false|   false| false|false|false|false|false|false| false|false| true|   false|
|      false|   false| false|fal

In [46]:
from pyspark.sql.functions import count, when

# 컬럼별 null 개수 (pandas: titanic_pdf.isna().sum())
display(titanic_sdf.select(
    [
        count(
            when(col(col_name).isNull(), col_name)
        ).alias(col_name)
        for col_name in titanic_sdf.columns
    ]
).toPandas())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,0,0,0,0,0,177,0,0,0,0,687,2


In [20]:
print(f"원래 데이터 프레임의 행 수: {titanic_sdf.count()}")
print(f"결측치(null 또는 NaN)가 하나라도 있는 행 제거 후: {titanic_sdf_3.dropna().count()}")  # Cabin 결측이 많아서 크게 줄어듦

원래 데이터 프레임의 행 수: 891


[Stage 20:>                                                         (0 + 1) / 1]

전체 데이터에서 결측치(null 또는 Nan) 제거 후 행 수: 183 


In [27]:
# 특정 컬럼에 결측치가 있는 행만 삭제
print(titanic_sdf_3.na.drop(subset=["Age", "Embarked"]).count())

# filter로 같은 결과
print(
    titanic_sdf_3.filter(
        (col("Age").isNotNull()) &
        (col("Embarked").isNotNull())
    ).count()
)

712
712


## 7. 결측치 채우기 (fillna / na.fill) - T

| 목적 | pandas | Spark |
|---|---|---|
| 전체 | `fillna(999)` | `fillna(999)` 또는 `na.fill(999)` |
| 특정 컬럼 | `df["Age"].fillna(999)` | `fillna(999, subset=["Age"])` |
| 컬럼별 다른 값 | `fillna({"Age": 30, "Cabin": "C000"})` | `fillna({"Age": 30, "Cabin": "C000"})` |
| 평균으로 채우기 | `fillna(df["Age"].mean())` | 평균을 값으로 꺼낸 뒤 `fillna(값, subset=[...])` |

- `value`와 **타입이 같은 컬럼에만** 적용됨: 숫자 -> 숫자형 컬럼, 문자열 -> 문자열 컬럼 (타입이 다른 컬럼은 조용히 무시)
- `value`에 DataFrame은 못 넣음. 집계 결과는 `first()[0]` 또는 `head()[0]`로 값을 꺼내서 사용

In [54]:
# subset 없이: 숫자 값은 숫자형 컬럼(Age)에만, 문자열 값은 문자열 컬럼(Cabin, Embarked)에만 적용
titanic_sdf.fillna(value=999).limit(10).show()
titanic_sdf.na.fill(value=999).limit(10).show()   # fillna와 같음

titanic_sdf.fillna(value="NA").limit(10).show()

+-----------+--------+------+--------------------+------+-----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex|  Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+-----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male| 22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female| 38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female| 26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female| 35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male| 35.0|    0|    0|          373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr. James|  male|999.0|    0|   

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25|   NA|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925|   NA|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05|   NA|       S|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|      

In [57]:
# subset으로 컬럼 지정
titanic_sdf.fillna(value=999, subset=["Age"]).limit(10).show()
titanic_sdf.fillna(value="NA", subset=["Cabin"]).limit(10).show()

+-----------+--------+------+--------------------+------+-----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex|  Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+-----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male| 22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female| 38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female| 26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female| 35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male| 35.0|    0|    0|          373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr. James|  male|999.0|    0|   

In [58]:
avg_age = titanic_sdf.select(F.avg(col("Age")))   # T: 결과는 1행 1열짜리 DataFrame
avg_age.show()

[Stage 97:>                                                         (0 + 1) / 1]

+-----------------+
|         avg(Age)|
+-----------------+
|29.69911764705882|
+-----------------+



In [59]:
titanic_sdf.fillna(value=avg_age, subset=["Age"])  # 오류: value에 DataFrame은 못 넣음

PySparkTypeError: [NOT_BOOL_OR_DICT_OR_FLOAT_OR_INT_OR_STR] Argument `value` should be a bool, dict, float, int or str, got DataFrame.

In [63]:
avg_age_value = avg_age.first()[0]   # A: Row -> 첫 번째 값(float)
titanic_sdf.fillna(value=avg_age_value, subset=["Age"]).show(10)

+-----------+--------+------+--------------------+------+-----------------+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex|              Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+-----------------+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|             22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|             38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|             26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|             35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|             35.0|    0|    0|          373450|

In [67]:
avg_age_value2 = avg_age.head()[0]   # head()도 n 없이 쓰면 Row 하나 반환 (first()와 같음)
avg_age_value2

29.69911764705882

In [68]:
# 컬럼별로 다른 값 채우기
titanic_sdf.fillna(
    {
        "Age": avg_age_value,
        "Cabin": "C000",
        "Embarked": "S"
    }
).show(10)

+-----------+--------+------+--------------------+------+-----------------+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex|              Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+-----------------+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|             22.0|    1|    0|       A/5 21171|   7.25| C000|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|             38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|             26.0|    0|    0|STON/O2. 3101282|  7.925| C000|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|             35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|             35.0|    0|    0|          373450|

## 8. 사용자 정의 함수 (UDF) vs when

| 방식 | 예시 | 특징 |
|---|---|---|
| pandas | `df["Age"].apply(func)` | 파이썬 함수를 행마다 적용 |
| Spark UDF | `udf(func, StringType())` | 파이썬 함수를 그대로 쓸 수 있음. 데이터를 파이썬으로 넘겨서 실행하므로 **느림** |
| Spark `when` | `when(조건, 값).when(...).otherwise(값)` | Spark 내장 함수라 **빠름**. 가능하면 이쪽 권장 |

- UDF에는 반환 타입을 지정해야 함 (`StringType()` 등)
- null은 UDF 안에 `None`으로 들어옴 -> `None <= 5` 비교에서 TypeError
- 해결: (1) 결측치를 먼저 채우기 (2) 함수 안에서 `None` 처리 (3) `when`에서 `isNull()`을 먼저 처리

In [69]:
def get_category(age):
    if age <= 5:
        return "Baby"
    elif age <= 12:
        return "Child"
    elif age <= 18:
        return "Teenager"
    elif age <= 25:
        return "Student"
    elif age <= 35:
        return "Young_Adult"
    elif age <= 60:
        return "Adult"
    else:
        return "Elderly"

In [71]:
# pandas 비교: apply
titanic_pdf["Age_category"] = titanic_pdf["Age"].apply(lambda x: get_category(x))
titanic_pdf.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Age_category
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,None,S,Student
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Adult
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,None,S,Young_Adult
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Young_Adult
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,None,S,Young_Adult
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,None,Q,Elderly
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,Adult
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,None,S,Baby
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,None,S,Young_Adult
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,None,C,Teenager


In [72]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

udf_get_category = udf(lambda x: get_category(x), StringType())  # udf(함수, 반환 타입)

In [74]:
titanic_sdf.withColumn("Age_category", udf_get_category(col("Age"))).show(10)  # 오류: Age가 null(None)인 행에서 TypeError

26/10/16 03:53:09 ERROR Executor: Exception in task 0.0 in stage 117.0 (TID 142)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/tmp/ipykernel_36/977695337.py", line 4, in <lambda>
  File "/tmp/ipykernel_36/2971306675.py", line 2, in get_category
TypeError: '<=' not supported between instances of 'NoneType' and 'int'

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:94)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:75)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:37)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:491)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at scala.coll

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/tmp/ipykernel_36/977695337.py", line 4, in <lambda>
  File "/tmp/ipykernel_36/2971306675.py", line 2, in get_category
TypeError: '<=' not supported between instances of 'NoneType' and 'int'


In [75]:
# 대안 1: 결측치 제거 혹은 replace 후 적용
titanic_sdf_filled = titanic_sdf.fillna(value = avg_age_value, subset = ["Age"])
titanic_sdf_filled.withColumn("Age_category", udf_get_category(col("Age"))).show(10)

+-----------+--------+------+--------------------+------+-----------------+-----+-----+----------------+-------+-----+--------+------------+
|PassengerId|Survived|Pclass|                Name|   Sex|              Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|Age_category|
+-----------+--------+------+--------------------+------+-----------------+-----+-----+----------------+-------+-----+--------+------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|             22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     Student|
|          2|       1|     1|Cumings, Mrs. Joh...|female|             38.0|    1|    0|        PC 17599|71.2833|  C85|       C|       Adult|
|          3|       1|     3|Heikkinen, Miss. ...|female|             26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S| Young_Adult|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|             35.0|    1|    0|          113803|   53.1| C123|       S| Young_Adult|
|          5|

In [78]:
# 대안 2: 함수 안에서 None 처리
def get_category_with_null_condition(age):
    if age is None:
        return "NA"
    if age <= 5:
        return "Baby"
    elif age <= 12:
        return "Child"
    elif age <= 18:
        return "Teenager"
    elif age <= 25:
        return "Student"
    elif age <= 35:
        return "Young_Adult"
    elif age <= 60:
        return "Adult"
    else:
        return "Elderly"

udf_get_category_with_null_condition = udf(lambda x: get_category_with_null_condition(x), StringType())
titanic_sdf.withColumn("Age_category", udf_get_category_with_null_condition(col("Age"))).show(10)

[Stage 119:>                                                        (0 + 1) / 1]

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+------------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|Age_category|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     Student|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|       Adult|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S| Young_Adult|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S| Young_Adult|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL

In [79]:
# 대안 3 (권장): UDF 대신 Spark 내장 when / otherwise
titanic_sdf.withColumn(
    "Age_category",
    when(col("Age").isNull(), "NA")
    .when(col("Age") <= 5, "Baby")
    .when(col("Age") <= 12, "Child")
    .when(col("Age") <= 18, "Teenage")
    .when(col("Age") <= 25, "Student")
    .when(col("Age") <= 35, "Young_Adult")
    .when(col("Age") <= 60, "Adult")
    .otherwise("Elderly")
).show(10)

[Stage 120:>                                                        (0 + 1) / 1]

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+------------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|Age_category|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     Student|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|       Adult|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S| Young_Adult|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S| Young_Adult|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL